# 🤖 Agentes Inteligentes: Simulación Completa en Python
> **Curso de IA | Russell & Norvig (2022)**  
> Simulación de los 5 tipos de agentes con datasets reales de Kaggle

---

## Estructura del notebook
| # | Tipo de Agente | Dataset Kaggle | Dominio |
|---|---|---|---|
| 1 | Reactivo Simple | Titanic (reglas fijas) | Clasificación |
| 2 | Basado en Modelos | House Prices (estado interno) | Regresión |
| 3 | Basado en Metas | Iris (búsqueda A*) | Planificación |
| 4 | Basado en Utilidad | Heart Disease (utilidad esperada) | Decisión médica |
| 5 | Agente que Aprende | MNIST / cualquier dataset | Aprendizaje RL |

**Bibliografía:**  
- Russell, S. & Norvig, P. (2022). *Artificial Intelligence: A Modern Approach* (4th ed.). Pearson.  
- Sutton, R. & Barto, A. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press.  
- Wooldridge, M. (2009). *An Introduction to MultiAgent Systems* (2nd ed.). Wiley.

In [ ]:
# ============================================================
#  INSTALACIÓN Y CONFIGURACIÓN
# ============================================================
!pip install pandas numpy scikit-learn matplotlib seaborn rich -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import deque, defaultdict
from dataclasses import dataclass, field
from typing import Any, List, Dict, Tuple, Optional
import random
import heapq
import warnings
warnings.filterwarnings('ignore')

# Estilo visual consistente
plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor': '#1a1a2e',
    'axes.edgecolor': '#444',
    'text.color': '#eee',
    'axes.labelcolor': '#eee',
    'xtick.color': '#aaa',
    'ytick.color': '#aaa',
    'grid.color': '#333',
    'font.family': 'monospace'
})
PALETTE = ['#00d4aa', '#7b68ee', '#ff6b6b', '#ffd93d', '#4ecdc4']

print('✅ Entorno listo.')

---
# PARTE 0 — Clase base: Agente Genérico
Todo agente en Russell & Norvig puede modelarse con esta interfaz mínima.

In [ ]:
# ============================================================
#  CLASE BASE ABSTRACTA: Agent
#  (Russell & Norvig 2022, Figura 2.1)
# ============================================================

class Agent:
    """
    Clase base para todos los agentes inteligentes.
    Implementa el ciclo percepcion → decision → accion.
    """
    def __init__(self, name: str):
        self.name = name
        self.perception_history: List[Any] = []   # secuencia de percepciones
        self.action_history: List[Any] = []
        self.performance_score: float = 0.0

    def perceive(self, percept: Any) -> None:
        """Registra una nueva percepcion del entorno."""
        self.perception_history.append(percept)

    def decide(self, percept: Any) -> Any:
        """Subclases implementan la logica de decision."""
        raise NotImplementedError

    def act(self, percept: Any) -> Any:
        """Ciclo completo: percibir → decidir → registrar accion."""
        self.perceive(percept)
        action = self.decide(percept)
        self.action_history.append(action)
        return action

    def report(self):
        print(f'\n{'='*55}')
        print(f'  AGENTE: {self.name}')
        print(f'  Percepciones procesadas : {len(self.perception_history)}')
        print(f'  Acciones ejecutadas     : {len(self.action_history)}')
        print(f'  Puntuacion de rendimiento: {self.performance_score:.4f}')
        print(f'{'='*55}')


class Environment:
    """Entorno generico que genera percepciones y evalua acciones."""
    def __init__(self, name: str):
        self.name = name
        self.state: Dict = {}
        self.step_count = 0

    def get_percept(self):
        raise NotImplementedError

    def evaluate(self, action: Any) -> float:
        raise NotImplementedError

print('✅ Clases base definidas: Agent, Environment')

---
# TIPO 1 — Agente Reactivo Simple
**Dataset:** Titanic (Kaggle)  
**Lógica:** Reglas condition → action fijas. No hay memoria ni planificación.
> *"If the percept is X, do action Y"* — Russell & Norvig (2022), fig. 2.9

In [ ]:
# ============================================================
#  AGENTE 1: REACTIVO SIMPLE — Titanic Survival Predictor
# ============================================================

# Cargar dataset Titanic (en Kaggle: /kaggle/input/titanic/train.csv)
# Para entorno local usamos el dataset de seaborn:
try:
    df = pd.read_csv('/kaggle/input/titanic/train.csv')
except FileNotFoundError:
    df = sns.load_dataset('titanic').rename(columns={
        'survived': 'Survived', 'pclass': 'Pclass',
        'sex': 'Sex', 'age': 'Age', 'fare': 'Fare'
    })
    df['Sex'] = df['Sex'].map({'male': 'male', 'female': 'female'})

df = df[['Survived','Pclass','Sex','Age','Fare']].dropna()
print(f'Dataset Titanic cargado: {df.shape[0]} pasajeros')
df.head(3)

In [ ]:
class SimpleReflexAgent(Agent):
    """
    Agente Reactivo Simple para predecir supervivencia en el Titanic.

    Arquitectura: CONDICION → ACCION (tabla de reglas)
    Sin memoria, sin planificacion, sin aprendizaje.

    Ref: Russell & Norvig (2022), sec. 2.4.2
    """

    RULES = [
        # (condicion_fn, prediccion, descripcion)
        (lambda p: p['Sex'] == 'female' and p['Pclass'] == 1,  1, 'Mujer 1a clase → SOBREVIVE'),
        (lambda p: p['Sex'] == 'female' and p['Pclass'] == 2,  1, 'Mujer 2a clase → SOBREVIVE'),
        (lambda p: p['Sex'] == 'female' and p['Pclass'] == 3
                   and p.get('Fare', 0) > 15,                  1, 'Mujer 3a clase tarifa alta → SOBREVIVE'),
        (lambda p: p['Sex'] == 'male'   and p.get('Age',30)<12, 1, 'Nino (menor 12) → SOBREVIVE'),
        (lambda p: p['Sex'] == 'male'   and p['Pclass'] == 1
                   and p.get('Fare', 0) > 50,                  1, 'Hombre 1a clase tarifa alta → SOBREVIVE'),
        (lambda p: True,                                        0, 'Default → NO SOBREVIVE'),
    ]

    def decide(self, percept: dict) -> int:
        """Evalua reglas en orden, retorna la primera que aplica."""
        for condition, action, _ in self.RULES:
            if condition(percept):
                return action
        return 0

    def evaluate_dataset(self, df: pd.DataFrame) -> dict:
        records = df.to_dict('records')
        predictions, actuals = [], []
        for row in records:
            pred = self.act(row)
            predictions.append(pred)
            actuals.append(row['Survived'])

        correct = sum(p == a for p, a in zip(predictions, actuals))
        self.performance_score = correct / len(actuals)
        return {'predictions': predictions, 'actuals': actuals,
                'accuracy': self.performance_score}


# ── Ejecutar agente ──
agent1 = SimpleReflexAgent('Agente Reactivo Simple — Titanic')
results1 = agent1.evaluate_dataset(df)
agent1.report()

# ── Visualización ──
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(results1['actuals'], results1['predictions'])
fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor='#0f0f0f')

# Matriz de confusion
sns.heatmap(cm, annot=True, fmt='d', ax=axes[0],
            cmap='RdYlGn', linewidths=1,
            xticklabels=['No sobrevive','Sobrevive'],
            yticklabels=['No sobrevive','Sobrevive'])
axes[0].set_title('Matriz de Confusión\nAgente Reactivo Simple', color='#eee', pad=12)
axes[0].set_xlabel('Predicho', color='#aaa')
axes[0].set_ylabel('Real', color='#aaa')

# Reglas disparadas
rule_counts = defaultdict(int)
for row in df.to_dict('records'):
    for cond, _, desc in SimpleReflexAgent.RULES:
        if cond(row):
            rule_counts[desc[:28]] += 1
            break

axes[1].barh(list(rule_counts.keys()), list(rule_counts.values()),
             color=PALETTE, edgecolor='#555')
axes[1].set_title('Frecuencia de disparo por regla', color='#eee', pad=12)
axes[1].set_xlabel('N° de veces activada', color='#aaa')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nAccuracy: {results1["accuracy"]:.2%}')
print(classification_report(results1['actuals'], results1['predictions'],
                             target_names=['No sobrevive', 'Sobrevive']))

---
# TIPO 2 — Agente Basado en Modelos
**Dataset:** House Prices (Kaggle)  
**Lógica:** Mantiene un **estado interno** (modelo del mercado inmobiliario) que actualiza con cada nueva percepción.
> *Mantiene un estado que depende de la historia perceptual* — Russell & Norvig (2022), fig. 2.11

In [ ]:
# ============================================================
#  AGENTE 2: BASADO EN MODELOS — House Price Estimator
# ============================================================
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# Dataset: en Kaggle usar /kaggle/input/house-prices-advanced-regression-techniques/train.csv
# Generamos un dataset sintético realista si no está disponible
try:
    hp = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
    hp = hp[['GrLivArea','OverallQual','GarageArea','YearBuilt','SalePrice']].dropna()
except FileNotFoundError:
    np.random.seed(42)
    n = 1000
    sqft = np.random.randint(500, 4000, n)
    quality = np.random.randint(1, 11, n)
    garage = np.random.randint(0, 1000, n)
    year = np.random.randint(1900, 2023, n)
    noise = np.random.normal(0, 15000, n)
    price = (sqft * 120 + quality * 20000 + garage * 50 +
             (year - 1900) * 800 + noise).clip(50000, 800000)
    hp = pd.DataFrame({
        'GrLivArea': sqft, 'OverallQual': quality,
        'GarageArea': garage, 'YearBuilt': year, 'SalePrice': price
    })

print(f'Dataset House Prices: {hp.shape[0]} casas')
hp.describe().round(0)

In [ ]:
@dataclass
class HouseMarketState:
    """Estado interno del modelo de mercado inmobiliario."""
    avg_price_per_sqft: float = 0.0
    market_trend: str = 'unknown'      # 'rising', 'falling', 'stable'
    n_observations: int = 0
    recent_prices: List[float] = field(default_factory=lambda: [])
    price_history: List[float] = field(default_factory=list)
    confidence: float = 0.0


class ModelBasedAgent(Agent):
    """
    Agente Basado en Modelos para estimacion de precios de casas.

    Mantiene dos modelos:
      1. Como evoluciona el mundo (tendencia del mercado)
      2. Como afectan sus acciones (impacto de la estimacion)

    Ref: Russell & Norvig (2022), sec. 2.4.3
    """

    def __init__(self, name: str, window: int = 50):
        super().__init__(name)
        self.state = HouseMarketState()
        self.window = window
        self.regressor = LinearRegression()
        self.scaler = StandardScaler()
        self.is_trained = False
        self.feature_cols = ['GrLivArea', 'OverallQual', 'GarageArea', 'YearBuilt']

    def update_world_model(self, percept: dict):
        """Actualiza el modelo del mundo con la nueva percepcion."""
        price = percept.get('SalePrice', 0)
        sqft = percept.get('GrLivArea', 1)

        self.state.n_observations += 1
        self.state.recent_prices.append(price)
        self.state.price_history.append(price)

        if len(self.state.recent_prices) > self.window:
            self.state.recent_prices.pop(0)

        self.state.avg_price_per_sqft = (
            np.mean(self.state.recent_prices) / max(sqft, 1)
        )

        # Detectar tendencia de mercado
        if len(self.state.recent_prices) >= 10:
            first_half = np.mean(self.state.recent_prices[:len(self.state.recent_prices)//2])
            second_half = np.mean(self.state.recent_prices[len(self.state.recent_prices)//2:])
            delta = (second_half - first_half) / max(first_half, 1)
            if delta > 0.02:
                self.state.market_trend = 'rising'
            elif delta < -0.02:
                self.state.market_trend = 'falling'
            else:
                self.state.market_trend = 'stable'

        # Confianza crece con el numero de observaciones
        self.state.confidence = min(1.0, self.state.n_observations / 200)

    def train_on_history(self, df: pd.DataFrame):
        """Entrena el regresor interno sobre los datos históricos acumulados."""
        X = df[self.feature_cols].values
        y = df['SalePrice'].values
        X_scaled = self.scaler.fit_transform(X)
        self.regressor.fit(X_scaled, y)
        self.is_trained = True

    def decide(self, percept: dict) -> dict:
        """Estima el precio ajustado por tendencia del mercado."""
        self.update_world_model(percept)

        if not self.is_trained:
            base_estimate = self.state.avg_price_per_sqft * percept.get('GrLivArea', 1500)
        else:
            features = np.array([[percept.get(c, 0) for c in self.feature_cols]])
            features_scaled = self.scaler.transform(features)
            base_estimate = float(self.regressor.predict(features_scaled)[0])

        # Ajuste por tendencia del mercado (modelo del mundo)
        trend_multiplier = {'rising': 1.03, 'falling': 0.97, 'stable': 1.0, 'unknown': 1.0}
        adjusted = base_estimate * trend_multiplier[self.state.market_trend]

        return {
            'estimated_price': adjusted,
            'market_trend': self.state.market_trend,
            'confidence': self.state.confidence,
            'n_obs': self.state.n_observations
        }


# ── Ejecutar agente ──
train_hp, test_hp = train_test_split(hp, test_size=0.3, random_state=42)

agent2 = ModelBasedAgent('Agente Basado en Modelos — House Prices')
agent2.train_on_history(train_hp)

decisions, actuals2, confidences, trends = [], [], [], []
for row in test_hp.to_dict('records'):
    result = agent2.act(row)
    decisions.append(result['estimated_price'])
    actuals2.append(row['SalePrice'])
    confidences.append(result['confidence'])
    trends.append(result['market_trend'])

mae = mean_absolute_error(actuals2, decisions)
r2  = r2_score(actuals2, decisions)
agent2.performance_score = r2
agent2.report()

# ── Visualización ──
fig, axes = plt.subplots(1, 3, figsize=(16, 4), facecolor='#0f0f0f')

# Real vs Predicho
axes[0].scatter(actuals2, decisions, alpha=0.5, c=PALETTE[1], s=20)
mn, mx = min(actuals2), max(actuals2)
axes[0].plot([mn,mx],[mn,mx], '--', c=PALETTE[0], lw=1.5)
axes[0].set_title(f'Real vs Estimado\nR²={r2:.3f}  MAE=${mae:,.0f}', color='#eee')
axes[0].set_xlabel('Precio real ($)', color='#aaa')
axes[0].set_ylabel('Estimado ($)', color='#aaa')

# Evolución de la confianza del agente
axes[1].plot(range(len(confidences)), confidences, c=PALETTE[2], lw=1.5)
axes[1].fill_between(range(len(confidences)), confidences, alpha=0.2, color=PALETTE[2])
axes[1].set_title('Confianza del estado interno\n(crece con experiencia)', color='#eee')
axes[1].set_xlabel('Percepciones acumuladas', color='#aaa')
axes[1].set_ylabel('Confianza [0–1]', color='#aaa')
axes[1].set_ylim(0, 1.05)

# Distribución de tendencias detectadas
trend_counts = pd.Series(trends).value_counts()
axes[2].bar(trend_counts.index, trend_counts.values,
            color=[PALETTE[0] if t=='rising' else PALETTE[2] if t=='falling' else PALETTE[3]
                   for t in trend_counts.index])
axes[2].set_title('Tendencia de mercado\ndetectada por el agente', color='#eee')
axes[2].set_ylabel('Frecuencia', color='#aaa')

plt.tight_layout()
plt.show()
print(f'MAE: ${mae:,.0f} | R²: {r2:.4f}')

---
# TIPO 3 — Agente Basado en Metas
**Dominio:** Navegación en grafo de ciudades (planificación A*)  
**Lógica:** Tiene una **meta** explícita y busca la secuencia de acciones para alcanzarla.
> *El agente considera el futuro: '¿qué pasará si hago X?' y '¿me acerca a mi meta?'* — Russell & Norvig (2022), sec. 2.4.4

In [ ]:
# ============================================================
#  AGENTE 3: BASADO EN METAS — Navegación con A*
#  (simula un agente GPS / sistema de planificacion de rutas)
# ============================================================

# Grafo de ciudades de Bolivia (distancias aproximadas en km)
CITIES = {
    'La Paz':       (16.5, -68.15),
    'Cochabamba':   (17.39, -66.16),
    'Santa Cruz':   (17.80, -63.17),
    'Oruro':        (17.97, -67.11),
    'Potosi':       (19.58, -65.75),
    'Sucre':        (19.04, -65.26),
    'Tarija':       (21.53, -64.73),
    'Trinidad':     (14.83, -64.90),
    'Cobija':       (11.03, -68.73),
}

ROADS = [
    ('La Paz', 'Oruro', 230),
    ('La Paz', 'Cochabamba', 390),
    ('Oruro', 'Cochabamba', 210),
    ('Oruro', 'Potosi', 330),
    ('Cochabamba', 'Santa Cruz', 480),
    ('Cochabamba', 'Sucre', 350),
    ('Potosi', 'Sucre', 160),
    ('Potosi', 'Tarija', 390),
    ('Sucre', 'Santa Cruz', 560),
    ('Sucre', 'Tarija', 440),
    ('Santa Cruz', 'Trinidad', 600),
    ('La Paz', 'Trinidad', 850),
    ('La Paz', 'Cobija', 1100),
]

def build_graph(roads):
    g = defaultdict(list)
    for a, b, d in roads:
        g[a].append((b, d))
        g[b].append((a, d))
    return g

GRAPH = build_graph(ROADS)


def haversine(c1, c2):
    """Distancia heuristica entre dos ciudades (en km)."""
    lat1, lon1 = CITIES[c1]
    lat2, lon2 = CITIES[c2]
    R = 6371
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1))*np.cos(np.radians(lat2))*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))


class GoalBasedAgent(Agent):
    """
    Agente Basado en Metas que usa el algoritmo A* para planificar rutas.

    Componentes:
      - Estado: ciudad actual
      - Meta: ciudad destino
      - Acciones: moverse a ciudad adyacente
      - Funcion de costo: distancia en km
      - Heuristica: distancia haversine al destino

    Ref: Russell & Norvig (2022), sec. 3.5 (A*)
    """

    def __init__(self, name: str, graph: dict):
        super().__init__(name)
        self.graph = graph
        self.current_city: Optional[str] = None
        self.goal: Optional[str] = None
        self.current_plan: List[str] = []
        self.nodes_explored = 0

    def set_goal(self, start: str, goal: str):
        self.current_city = start
        self.goal = goal
        self.current_plan = []
        self.nodes_explored = 0

    def a_star(self) -> Tuple[List[str], float]:
        """Busqueda A* desde ciudad actual hasta meta."""
        start = self.current_city
        goal  = self.goal

        # (f_cost, g_cost, node, path)
        heap = [(0, 0, start, [start])]
        visited = {}

        while heap:
            f, g, node, path = heapq.heappop(heap)
            self.nodes_explored += 1

            if node in visited and visited[node] <= g:
                continue
            visited[node] = g

            if node == goal:
                return path, g

            for neighbor, cost in self.graph[node]:
                new_g = g + cost
                h     = haversine(neighbor, goal)
                heapq.heappush(heap, (new_g + h, new_g, neighbor, path + [neighbor]))

        return [], float('inf')

    def decide(self, percept: dict) -> dict:
        """Genera un plan A* para llegar a la meta."""
        start = percept['start']
        goal  = percept['goal']
        self.set_goal(start, goal)

        path, total_cost = self.a_star()
        self.current_plan = path
        return {
            'path': path,
            'total_km': total_cost,
            'nodes_explored': self.nodes_explored,
            'steps': len(path) - 1
        }


# ── Ejecutar múltiples misiones ──
agent3 = GoalBasedAgent('Agente Basado en Metas — Navegación Bolivia', GRAPH)

missions = [
    ('La Paz', 'Tarija'),
    ('Cochabamba', 'Cobija'),
    ('Oruro', 'Trinidad'),
    ('Potosi', 'La Paz'),
    ('Santa Cruz', 'Cobija'),
]

print('\n MISIONES DEL AGENTE BASADO EN METAS (A*)\n')
print(f'{'Origen':<15} {'Destino':<15} {'Ruta':<45} {'km':>7} {'Nodos':>7}')
print('-'*95)

mission_results = []
for start, goal in missions:
    result = agent3.act({'start': start, 'goal': goal})
    route = ' → '.join(result['path'])
    print(f'{start:<15} {goal:<15} {route:<45} {result["total_km"]:>7.0f} {result["nodes_explored"]:>7}')
    mission_results.append(result)

agent3.performance_score = 1.0  # planificador exacto
agent3.report()

# ── Visualización del grafo con la última ruta ──
fig, ax = plt.subplots(figsize=(10, 8), facecolor='#0f0f0f')
ax.set_facecolor('#1a1a2e')

# Dibujar todas las carreteras
for a, b, d in ROADS:
    la, lo_a = CITIES[a]
    lb, lo_b = CITIES[b]
    ax.plot([-lo_a, -lo_b], [-la, -lb], '-', color='#444', lw=1, zorder=1)
    mx, my = (-lo_a - lo_b)/2, (-la - lb)/2
    ax.text(mx, my, f'{d}km', fontsize=7, color='#666', ha='center')

# Resaltar ruta La Paz → Tarija
highlight = mission_results[0]['path']
for i in range(len(highlight)-1):
    la, lo_a = CITIES[highlight[i]]
    lb, lo_b = CITIES[highlight[i+1]]
    ax.plot([-lo_a, -lo_b], [-la, -lb], '-', color=PALETTE[0], lw=3, zorder=2)

# Ciudades
for city, (lat, lon) in CITIES.items():
    color = PALETTE[0] if city in highlight else PALETTE[1]
    size  = 120 if city in [highlight[0], highlight[-1]] else 60
    ax.scatter(-lon, -lat, s=size, c=color, zorder=3, edgecolors='white', linewidths=0.5)
    ax.text(-lon + 0.1, -lat + 0.1, city, fontsize=9,
            color='white' if city in highlight else '#aaa', fontweight='bold')

ax.set_title(f'Ruta A*: La Paz → Tarija ({mission_results[0]["total_km"]:.0f} km)\nNodos explorados: {mission_results[0]["nodes_explored"]}',
             color='#eee', pad=12)
ax.set_xlabel('Longitud (invertida)', color='#aaa')
ax.set_ylabel('Latitud (invertida)', color='#aaa')
plt.tight_layout()
plt.show()

---
# TIPO 4 — Agente Basado en Utilidad
**Dataset:** Heart Disease (Kaggle / UCI)  
**Lógica:** No solo busca una meta; maximiza una **función de utilidad** que pondera múltiples criterios bajo incertidumbre.
> *Utility is a function that maps a state to a real number that describes the associated degree of happiness.* — Russell & Norvig (2022), sec. 2.4.5

In [ ]:
# ============================================================
#  AGENTE 4: BASADO EN UTILIDAD — Decisión médica
#  El agente decide el tratamiento que maximiza utilidad esperada
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV

# Dataset Heart Disease (UCI via Kaggle: /kaggle/input/heart-disease-uci/heart.csv)
try:
    heart = pd.read_csv('/kaggle/input/heart-disease-uci/heart.csv')
except FileNotFoundError:
    # Dataset sintético basado en distribuciones UCI reales
    np.random.seed(99)
    n = 800
    age     = np.random.randint(28, 78, n)
    cp      = np.random.randint(0, 4, n)
    trestbps= np.random.randint(90, 200, n)
    chol    = np.random.randint(150, 400, n)
    thalach = np.random.randint(70, 200, n)
    oldpeak = np.random.uniform(0, 5, n)
    target  = ((age > 55) & (cp > 1) & (thalach < 140) |
               (chol > 300) & (trestbps > 150)).astype(int)
    noise_mask = np.random.rand(n) < 0.1
    target[noise_mask] = 1 - target[noise_mask]
    heart = pd.DataFrame({
        'age':age,'cp':cp,'trestbps':trestbps,'chol':chol,
        'thalach':thalach,'oldpeak':oldpeak,'target':target
    })

print(f'Dataset Heart Disease: {heart.shape[0]} pacientes')
print(f'Tasa de enfermedad: {heart.target.mean():.1%}')

In [ ]:
@dataclass
class MedicalAction:
    name: str
    cost: float          # costo economico normalizado [0,1]
    risk: float          # riesgo de efectos adversos [0,1]
    efficacy_if_ill: float   # efectividad si hay enfermedad [0,1]
    efficacy_if_healthy: float  # "utilidad" si no hay enfermedad [0,1]


MEDICAL_ACTIONS = [
    MedicalAction('Observacion y seguimiento', cost=0.05, risk=0.02,
                  efficacy_if_ill=0.30, efficacy_if_healthy=0.95),
    MedicalAction('Medicacion antihipertensiva', cost=0.25, risk=0.10,
                  efficacy_if_ill=0.65, efficacy_if_healthy=0.70),
    MedicalAction('Pruebas de estres cardiaco', cost=0.40, risk=0.05,
                  efficacy_if_ill=0.80, efficacy_if_healthy=0.80),
    MedicalAction('Cateterismo + tratamiento', cost=0.85, risk=0.35,
                  efficacy_if_ill=0.92, efficacy_if_healthy=0.30),
]


class UtilityBasedAgent(Agent):
    """
    Agente Basado en Utilidad para decision medica.

    Calcula la Utilidad Esperada (EU) para cada accion:
      EU(a) = P(ill|percept) * U(a, ill) + P(healthy|percept) * U(a, healthy)
    
    Donde U(a, s) combina eficacia, costo y riesgo con pesos ajustables.

    Ref: Russell & Norvig (2022), sec. 16.1 — Maximum Expected Utility
         Wooldridge (2009), cap. 4
    """

    def __init__(self, name: str, actions: List[MedicalAction],
                 w_efficacy=0.6, w_cost=0.2, w_risk=0.2):
        super().__init__(name)
        self.actions = actions
        self.w_efficacy = w_efficacy
        self.w_cost = w_cost
        self.w_risk = w_risk
        self.feature_cols = ['age','cp','trestbps','chol','thalach','oldpeak']
        self.classifier = None
        self.utility_log: List[dict] = []

    def train(self, df: pd.DataFrame):
        X = df[self.feature_cols]
        y = df['target']
        base = RandomForestClassifier(n_estimators=100, random_state=42)
        self.classifier = CalibratedClassifierCV(base, cv=3)
        self.classifier.fit(X, y)

    def state_utility(self, action: MedicalAction, p_ill: float) -> float:
        """EU(a) ponderado por eficacia, costo y riesgo."""
        u_if_ill     = (self.w_efficacy * action.efficacy_if_ill
                        - self.w_cost  * action.cost
                        - self.w_risk  * action.risk)
        u_if_healthy = (self.w_efficacy * action.efficacy_if_healthy
                        - self.w_cost  * action.cost
                        - self.w_risk  * action.risk * 0.3)
        return p_ill * u_if_ill + (1 - p_ill) * u_if_healthy

    def decide(self, percept: dict) -> dict:
        """Elige la accion con maxima utilidad esperada."""
        features = pd.DataFrame([{c: percept.get(c, 0) for c in self.feature_cols}])
        p_ill = float(self.classifier.predict_proba(features)[0][1])

        utilities = [(self.state_utility(a, p_ill), a) for a in self.actions]
        best_utility, best_action = max(utilities, key=lambda x: x[0])

        decision = {
            'action': best_action.name,
            'p_disease': p_ill,
            'expected_utility': best_utility,
            'all_utilities': {a.name: u for u, a in utilities}
        }
        self.utility_log.append(decision)
        return decision


# ── Ejecutar agente ──
train_h, test_h = train_test_split(heart, test_size=0.3, random_state=42)
agent4 = UtilityBasedAgent('Agente Basado en Utilidad — Decisión Médica', MEDICAL_ACTIONS)
agent4.train(train_h)

decisions4, actuals4 = [], []
for row in test_h.to_dict('records'):
    result = agent4.act(row)
    decisions4.append(result)
    actuals4.append(row['target'])

agent4.performance_score = np.mean([d['expected_utility'] for d in decisions4])
agent4.report()

# ── Visualización ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor='#0f0f0f')

# Distribución de P(enfermedad)
probs_ill  = [d['p_disease'] for d in decisions4]
probs_healthy = [1-p for p in probs_ill]
c_actual = [PALETTE[2] if a==1 else PALETTE[0] for a in actuals4]
axes[0].scatter(range(len(probs_ill)), sorted(probs_ill), c=sorted(c_actual), s=15, alpha=0.7)
axes[0].axhline(0.5, ls='--', c=PALETTE[3], lw=1)
axes[0].set_title('P(enfermedad) estimada\npor paciente', color='#eee')
axes[0].set_ylabel('Probabilidad', color='#aaa')
patch1 = mpatches.Patch(color=PALETTE[2], label='Enfermo (real)')
patch2 = mpatches.Patch(color=PALETTE[0], label='Sano (real)')
axes[0].legend(handles=[patch1, patch2], fontsize=9)

# Acciones elegidas por el agente
action_counts = pd.Series([d['action'] for d in decisions4]).value_counts()
axes[1].barh(action_counts.index, action_counts.values,
             color=PALETTE[:len(action_counts)], edgecolor='#555')
axes[1].set_title('Acciones elegidas\n(decisiones del agente)', color='#eee')
axes[1].set_xlabel('N° de pacientes', color='#aaa')

# Utilidades esperadas por acción para un caso ejemplo
example = decisions4[0]
action_names = [a[:20] for a in example['all_utilities'].keys()]
action_utils = list(example['all_utilities'].values())
bars = axes[2].bar(range(len(action_names)), action_utils,
                   color=[PALETTE[0] if u == max(action_utils) else PALETTE[4]
                          for u in action_utils])
axes[2].set_xticks(range(len(action_names)))
axes[2].set_xticklabels(action_names, rotation=20, ha='right', fontsize=8)
axes[2].set_title(f'Utilidad esperada por acción\nP(ill)={example["p_disease"]:.2f}', color='#eee')
axes[2].set_ylabel('Utilidad esperada', color='#aaa')

plt.tight_layout()
plt.show()

---
# TIPO 5 — Agente que Aprende (Learning Agent)
**Dominio:** Grid World con Q-Learning (Reinforcement Learning)  
**Lógica:** El agente no conoce las reglas del entorno; aprende por prueba y error acumulando experiencia.
> *A learning agent can be divided into four conceptual components: learning element, performance element, critic, and problem generator.* — Russell & Norvig (2022), sec. 19.1

In [ ]:
# ============================================================
#  AGENTE 5: AGENTE QUE APRENDE — Q-Learning en Grid World
#  (simulacion de entorno tipo Kaggle Gym / gymnasium)
# ============================================================

class GridWorld:
    """
    Entorno Grid World 6x6.
    El agente debe llegar de (0,0) a (5,5) evitando obstaculos.
    """
    ACTIONS = {0: (-1,0), 1: (1,0), 2: (0,-1), 3: (0,1)}  # arriba, abajo, izq, der
    ACTION_NAMES = {0:'↑', 1:'↓', 2:'←', 3:'→'}

    def __init__(self, size=6, n_obstacles=7, seed=42):
        self.size = size
        np.random.seed(seed)
        self.start = (0, 0)
        self.goal  = (size-1, size-1)
        all_cells = [(r,c) for r in range(size) for c in range(size)]
        all_cells.remove(self.start)
        all_cells.remove(self.goal)
        self.obstacles = set(map(tuple, [all_cells[i] for i in
                                         np.random.choice(len(all_cells), n_obstacles, replace=False)]))
        self.state = self.start

    def reset(self):
        self.state = self.start
        return self.state

    def step(self, action: int) -> Tuple[tuple, float, bool]:
        dr, dc = self.ACTIONS[action]
        nr, nc = self.state[0]+dr, self.state[1]+dc

        # Fuera de límites o en obstáculo → penalizar y quedarse
        if not (0 <= nr < self.size and 0 <= nc < self.size):
            return self.state, -1.0, False
        if (nr, nc) in self.obstacles:
            return self.state, -5.0, False

        self.state = (nr, nc)
        if self.state == self.goal:
            return self.state, +100.0, True
        return self.state, -0.5, False  # pequeña penalización por paso


class QLearningAgent(Agent):
    """
    Agente que aprende mediante Q-Learning (Sutton & Barto, 2018).

    Componentes segun Russell & Norvig (2022):
      - Elemento de rendimiento: politica epsilon-greedy
      - Critico: reward del entorno
      - Elemento de aprendizaje: actualizacion Q(s,a)
      - Generador de problemas: exploracion epsilon

    Q(s,a) ← Q(s,a) + α[r + γ·max_a' Q(s',a') - Q(s,a)]
    """

    def __init__(self, name: str, n_actions: int,
                 alpha=0.1, gamma=0.95, epsilon=1.0, epsilon_decay=0.995):
        super().__init__(name)
        self.n_actions = n_actions
        self.alpha = alpha       # tasa de aprendizaje
        self.gamma = gamma       # factor de descuento
        self.epsilon = epsilon   # exploracion inicial
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = 0.05
        self.Q = defaultdict(lambda: np.zeros(n_actions))  # tabla Q
        self.episode_rewards: List[float] = []
        self.episode_steps:   List[int]   = []

    def decide(self, state) -> int:
        """Politica epsilon-greedy: explorar o explotar."""
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)  # explorar
        return int(np.argmax(self.Q[state]))           # explotar

    def learn(self, s, a, r, s_next, done):
        """Actualizacion Q-Learning (off-policy TD control)."""
        best_next = 0 if done else np.max(self.Q[s_next])
        td_target = r + self.gamma * best_next
        td_error  = td_target - self.Q[s][a]
        self.Q[s][a] += self.alpha * td_error

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)


# ── Entrenamiento ──
env = GridWorld(size=6, n_obstacles=7)
agent5 = QLearningAgent('Agente Q-Learning — Grid World', n_actions=4)

N_EPISODES = 1500
EVAL_EVERY = 100
eval_rewards = []

for ep in range(N_EPISODES):
    state = env.reset()
    total_reward = 0
    steps = 0

    for _ in range(200):  # max pasos por episodio
        action = agent5.act(state)
        next_state, reward, done = env.step(action)
        agent5.learn(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward
        steps += 1
        if done:
            break

    agent5.decay_epsilon()
    agent5.episode_rewards.append(total_reward)
    agent5.episode_steps.append(steps)

    if (ep + 1) % EVAL_EVERY == 0:
        avg = np.mean(agent5.episode_rewards[-EVAL_EVERY:])
        eval_rewards.append(avg)
        print(f'Ep {ep+1:>5} | Avg reward: {avg:>8.1f} | ε={agent5.epsilon:.3f}')

agent5.performance_score = np.mean(agent5.episode_rewards[-100:])
agent5.report()

In [ ]:
# ── Visualización del aprendizaje + política aprendida ──
fig, axes = plt.subplots(1, 3, figsize=(17, 5), facecolor='#0f0f0f')

# Curva de aprendizaje
window = 50
smoothed = pd.Series(agent5.episode_rewards).rolling(window).mean()
axes[0].plot(agent5.episode_rewards, alpha=0.2, c=PALETTE[4], lw=0.5)
axes[0].plot(smoothed, c=PALETTE[0], lw=2)
axes[0].set_title(f'Curva de aprendizaje\n(media móvil {window} eps)', color='#eee')
axes[0].set_xlabel('Episodio', color='#aaa')
axes[0].set_ylabel('Recompensa total', color='#aaa')
axes[0].grid(alpha=0.2)

# Política aprendida (flecha en cada celda)
size = env.size
grid_colors = np.zeros((size, size, 3))
ARROW = {0:'↑', 1:'↓', 2:'←', 3:'→'}

for r in range(size):
    for c in range(size):
        if (r,c) in env.obstacles:
            axes[1].add_patch(plt.Rectangle((c,size-1-r), 1, 1,
                                             color='#ff6b6b', alpha=0.7))
        elif (r,c) == env.goal:
            axes[1].add_patch(plt.Rectangle((c,size-1-r), 1, 1,
                                             color='#00d4aa', alpha=0.7))
            axes[1].text(c+0.5, size-1-r+0.5, 'META', ha='center', va='center',
                         color='white', fontsize=9, fontweight='bold')
        elif (r,c) == env.start:
            axes[1].add_patch(plt.Rectangle((c,size-1-r), 1, 1,
                                             color='#7b68ee', alpha=0.5))
            axes[1].text(c+0.5, size-1-r+0.5, 'INICIO', ha='center', va='center',
                         color='white', fontsize=8)
        else:
            best_a = int(np.argmax(agent5.Q[(r,c)]))
            qval = max(agent5.Q[(r,c)])
            intensity = min(1.0, max(0, (qval + 5) / 110))
            axes[1].add_patch(plt.Rectangle((c,size-1-r), 1, 1,
                                             color=(0, intensity*0.6, intensity*0.5), alpha=0.5))
            axes[1].text(c+0.5, size-1-r+0.5, ARROW[best_a],
                         ha='center', va='center', color='white', fontsize=14)

axes[1].set_xlim(0, size); axes[1].set_ylim(0, size)
axes[1].set_xticks(range(size)); axes[1].set_yticks(range(size))
axes[1].grid(True, color='#555', lw=0.5)
axes[1].set_title('Política aprendida (Q-Learning)\nFlechas = acción óptima por celda', color='#eee')

# Episodios necesarios para llegar a la meta
steps_smooth = pd.Series(agent5.episode_steps).rolling(50).mean()
axes[2].plot(agent5.episode_steps, alpha=0.15, c=PALETTE[3], lw=0.5)
axes[2].plot(steps_smooth, c=PALETTE[3], lw=2)
axes[2].set_title('Pasos por episodio\n(menos = más eficiente)', color='#eee')
axes[2].set_xlabel('Episodio', color='#aaa')
axes[2].set_ylabel('Pasos hasta llegar a meta', color='#aaa')
axes[2].grid(alpha=0.2)

plt.tight_layout()
plt.show()

# ── Demostración: el agente sigue la política aprendida ──
state = env.reset()
agent5.epsilon = 0  # solo explotación
path_demo = [state]
total_r = 0
for _ in range(50):
    action = agent5.decide(state)
    next_state, reward, done = env.step(action)
    path_demo.append(next_state)
    total_r += reward
    state = next_state
    if done:
        break

print(f'\nDemostracion de la politica aprendida:')
print(f'Ruta: {" → ".join(str(p) for p in path_demo)}')
print(f'Pasos: {len(path_demo)-1} | Recompensa total: {total_r:.1f}')
print(f'Exito: {"SI" if path_demo[-1] == env.goal else "NO"}')

---
# COMPARATIVA FINAL: Los 5 agentes
Resumen cuantitativo de rendimiento, complejidad y uso de recursos.

In [ ]:
# ============================================================
#  PANEL COMPARATIVO FINAL
# ============================================================

summary = pd.DataFrame([
    {
        'Tipo': '1. Reactivo simple',
        'Agente': agent1.name.split('—')[0].strip(),
        'Dataset': 'Titanic',
        'Metrica': f'Accuracy: {agent1.performance_score:.2%}',
        'Memoria': 'No', 'Planificacion': 'No', 'Aprendizaje': 'No',
        'Complejidad': 'O(R)',
    },
    {
        'Tipo': '2. Basado en modelos',
        'Agente': agent2.name.split('—')[0].strip(),
        'Dataset': 'House Prices',
        'Metrica': f'R²: {agent2.performance_score:.4f}',
        'Memoria': 'Si', 'Planificacion': 'No', 'Aprendizaje': 'Parcial',
        'Complejidad': 'O(N·F)',
    },
    {
        'Tipo': '3. Basado en metas',
        'Agente': agent3.name.split('—')[0].strip(),
        'Dataset': 'Ciudades Bolivia',
        'Metrica': f'Ruta optima garantizada',
        'Memoria': 'Si', 'Planificacion': 'Si', 'Aprendizaje': 'No',
        'Complejidad': 'O(b^d)',
    },
    {
        'Tipo': '4. Basado en utilidad',
        'Agente': agent4.name.split('—')[0].strip(),
        'Dataset': 'Heart Disease',
        'Metrica': f'EU media: {agent4.performance_score:.3f}',
        'Memoria': 'Si', 'Planificacion': 'Si', 'Aprendizaje': 'Si',
        'Complejidad': 'O(A·S)',
    },
    {
        'Tipo': '5. Agente que aprende',
        'Agente': agent5.name.split('—')[0].strip(),
        'Dataset': 'Grid World (RL)',
        'Metrica': f'Avg reward: {agent5.performance_score:.1f}',
        'Memoria': 'Si', 'Planificacion': 'Si', 'Aprendizaje': 'Si',
        'Complejidad': 'O(S·A·T)',
    },
])

print('\n' + '='*100)
print('  RESUMEN COMPARATIVO — 5 TIPOS DE AGENTES INTELIGENTES')
print('='*100)
print(summary[['Tipo','Dataset','Metrica','Memoria','Planificacion','Aprendizaje','Complejidad']].to_string(index=False))
print('='*100)
print('\nRef: R=reglas, N=datos, F=features, b=factor ramificación, d=prof., A=acciones, S=estados, T=episodios')
print('Russell & Norvig (2022), cap. 2 | Sutton & Barto (2018) | Wooldridge (2009)')

# Radar chart comparativo
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True), facecolor='#0f0f0f')
ax.set_facecolor('#1a1a2e')

criteria = ['Memoria', 'Planificacion', 'Aprendizaje', 'Velocidad\ndecision', 'Transparencia', 'Generalidad']
scores = [
    [0, 0, 0, 5, 5, 1],   # Reactivo simple
    [4, 0, 2, 4, 4, 2],   # Basado en modelos
    [3, 5, 0, 2, 4, 3],   # Basado en metas
    [4, 4, 3, 2, 3, 4],   # Basado en utilidad
    [5, 5, 5, 1, 1, 5],   # Agente que aprende
]
labels = ['Reactivo simple','Mod. modelos','Mod. metas','Mod. utilidad','Aprende']

N = len(criteria)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

for i, (score, label) in enumerate(zip(scores, labels)):
    values = score + score[:1]
    ax.plot(angles, values, 'o-', lw=1.5, c=PALETTE[i], label=label)
    ax.fill(angles, values, alpha=0.08, c=PALETTE[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(criteria, color='#ccc', fontsize=10)
ax.set_yticks([1,2,3,4,5])
ax.set_yticklabels(['1','2','3','4','5'], color='#666', fontsize=8)
ax.set_ylim(0, 5)
ax.grid(color='#333', linewidth=0.5)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15),
          facecolor='#1a1a2e', edgecolor='#444', labelcolor='#eee', fontsize=9)
ax.set_title('Perfil comparativo\nde los 5 tipos de agente', color='#eee',
             fontsize=13, pad=20)

plt.tight_layout()
plt.show()

print('\n✅ Notebook completado. Todos los agentes simulados con exito.')

---
# 💡 Ideas para Notebooks Nuevos en Kaggle

| # | Título | Dataset | Tipo de Agente | Técnica |
|---|--------|---------|----------------|---------|
| 1 | **Agente BDI para diagnóstico médico** | MIMIC-III | Deliberativo | Prolog / Lógica primer orden |
| 2 | **Multi-Agente de trading financiero** | Crypto prices | Utilidad + Aprende | MARL, PPO |
| 3 | **Agente reactivo para moderación NLP** | Jigsaw Toxic | Reactivo | Reglas + BERT |
| 4 | **Agente navegador con LLM (LangChain)** | Wikipedia API | Aprende | RAG + Tool use |
| 5 | **Simulación multiagente de epidemias** | COVID-19 data | Multi-agente | Mesa framework |
| 6 | **Agente de recomendación contextual** | MovieLens | Utilidad | MDP + Bandits |
| 7 | **Agente para optimización de carteras** | S&P 500 stocks | Utilidad | Portfolio + RL |
| 8 | **Agente que aprende a jugar Blackjack** | Gymnasium | Aprende (RL) | Monte Carlo / TD |

**Referencia general:** Russell & Norvig (2022), caps. 2–19 | Sutton & Barto (2018) | Wooldridge (2009)